# LATTICE — Exploratory Notebook
**ArXivist-generated** | Paper: [arXiv:2607.14410](https://arxiv.org/abs/2607.14410)

Visualizes LATTICE embeddings: spatial domain structure (Figure 4-style),
modality-ladder behavior (Figure 2-style), and the effect of masking ratio
on reconstruction quality.

> **Requires a trained checkpoint.** Run `python train.py --config configs/config.yaml`
> first (or `--debug` for a quick one), then point `CHECKPOINT_PATH` below at it.


In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import torch
import matplotlib.pyplot as plt

CHECKPOINT_PATH = "../checkpoints/best.pt"  # update if needed


In [ ]:
from pathlib import Path
from lattice.utils.config import load_config
from lattice.data.dataset import SyntheticSpatialMultimodalDataset
from lattice.models.lattice_model import LatticeModel

config = load_config("../configs/config.yaml")

if not Path(CHECKPOINT_PATH).exists():
    print(f"No checkpoint found at {CHECKPOINT_PATH}.")
    print("Run `python train.py --config configs/config.yaml --debug` from the repo root first.")
else:
    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}")


## Load data + model


In [ ]:
try:
    dataset = SyntheticSpatialMultimodalDataset(
        num_samples=config["data"]["num_synthetic_samples"],
        spots_per_sample=config["data"]["spots_per_sample"],
        gene_count_range=tuple(config["data"]["gene_count_range"]),
        num_modality_blocks=config["model"]["num_modality_blocks"],
        seed=config["training"]["seed"],
    )
    model = LatticeModel(
        modality_dims=dataset.modality_dims,
        hidden_dim=config["model"]["hidden_dim"],
        graph_num_layers=config["model"]["graph_encoder"]["num_layers"],
        graph_num_heads=config["model"]["graph_encoder"]["num_heads"],
        graph_dropout=config["model"]["graph_encoder"]["dropout"],
        decoder_hidden_width=config["model"]["decoder"]["hidden_width"],
        proj_hidden_width=config["model"]["projection_head"]["hidden_width"],
        proj_output_dim=config["model"]["projection_head"]["output_dim"],
        knn_k=config["data"]["knn_k"],
        edge_weight_mode=config["data"]["edge_weight_mode"],
    )
    if Path(CHECKPOINT_PATH).exists():
        model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(model)
except Exception as e:
    print(f"Setup failed: {e}")
    print("Make sure you've run train.py at least once (--debug is fine).")


## Visualization 1 — Spatial domains on tissue coordinates

Analogous to the paper's Figure 4 (Leiden clusters over tissue coordinates).
Colors here are the SYNTHETIC ground-truth domain labels used to generate
this sample (not a paper-reported result).


In [ ]:
try:
    sample = dataset[0]
    coords = sample["coords"].numpy()
    domain_labels = sample["domain_labels"].numpy()

    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(coords[:, 0], coords[:, 1], c=domain_labels, cmap="tab10", s=8)
    plt.title("Synthetic spatial domains (sample 0)")
    plt.xlabel("x"); plt.ylabel("y")
    plt.colorbar(scatter, label="domain id")
    plt.gca().set_aspect("equal")
    plt.show()
except Exception as e:
    print(f"Visualization failed: {e}")


## Visualization 2 — Embedding structure via PCA

Analogous to the paper's Figure 3 (joint PCA). Colors by synthetic domain to
check whether the learned embedding separates the domains it should.


In [ ]:
try:
    with torch.no_grad():
        z = model.embed(sample["modality_blocks"], sample["presence_mask"], sample["coords"]).numpy()

    z_centered = z - z.mean(axis=0, keepdims=True)
    u, s, vt = np.linalg.svd(z_centered, full_matrices=False)
    z_pca = u[:, :2] * s[:2]

    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(z_pca[:, 0], z_pca[:, 1], c=domain_labels, cmap="tab10", s=8)
    plt.title("Embedding PCA, colored by synthetic domain")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.colorbar(scatter, label="domain id")
    plt.show()
except Exception as e:
    print(f"Visualization failed: {e}")


## Visualization 3 — Modality-ladder ablation (M1 vs M5)

Reproduces the *shape* of the paper's Figure 2 (modality ladder trend) on
synthetic data: embed the same sample with only modality 1 active (M1) vs.
all 5 active (M5), and compare embedding silhouette against the synthetic
domain labels.


In [ ]:
try:
    from lattice.evaluation.metrics import embedding_silhouette
    from sklearn.cluster import KMeans

    def embed_with_active(active_flags):
        blocks = [
            blk if is_on else torch.zeros_like(blk)
            for blk, is_on in zip(sample["modality_blocks"], active_flags)
        ]
        pm = sample["presence_mask"].clone()
        for m, is_on in enumerate(active_flags):
            if not is_on:
                pm[:, m] = 0.0
        with torch.no_grad():
            return model.embed(blocks, pm, sample["coords"]).numpy()

    z_m1 = embed_with_active([True, False, False, False, False])
    z_m5 = embed_with_active([True, True, True, True, True])

    labels_m1 = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(z_m1)
    labels_m5 = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(z_m5)

    sil_m1 = embedding_silhouette(z_m1, labels_m1)
    sil_m5 = embedding_silhouette(z_m5, labels_m5)

    plt.figure(figsize=(4, 4))
    plt.bar(["M1", "M5"], [sil_m1, sil_m5], color=["#888", "#2a6"])
    plt.ylabel("Embedding silhouette")
    plt.title("Modality-ladder effect (this sample only)")
    plt.show()
    print(f"M1 silhouette: {sil_m1:.3f}  |  M5 silhouette: {sil_m5:.3f}")
    print("Paper's cohort-level trend (Table 2): LATTICE M1 silhouette=0.147, M5 silhouette=0.417")
except Exception as e:
    print(f"Visualization failed: {e}")


## Next steps

- Try `--modality-level M2`/`M3`/`M4` in `evaluate.py` to reproduce the full ladder on your data.
- Swap in real data via the interface documented in `data/README_data.md` for numbers comparable to Table 2/3.
